# 02 - Inference Qwen2-VL (baseline / adapter) trên Kaggle

**Mục tiêu:** chạy inference trên split `test` của `bbdontcry/vietnamese-image-captioning` (detail caption).

**Inputs**
- Dataset: `bbdontcry/vietnamese-image-captioning` (split `test`)
- Chọn 1 trong 2 kiểu chạy (`RUN_KIND`):
  - `RUN_KIND="baseline"`: dùng base model `Qwen/Qwen2-VL-2B-Instruct`  → `RUN_ID="baseline"`
  - `RUN_KIND="adapter"`: dùng `run_config.json` + thư mục `adapter/` xuất từ notebook 01  
    → `RUN_ID` được đọc từ `run_config.json` (ví dụ: `A`, `B`, `demo`)

**Outputs** *(trong `OUT_DIR=/kaggle/working/`)*  
- `{RUN_ID}_predictions_test_detail.csv` : gồm `id`, `gt_detail`, `pred_detail`, metadata chạy
- `{RUN_ID}_inference_config.json` : log lại prompt + gen config để tái lập kết quả

**Notebook kế tiếp:** chạy `03a-qwen-vl-metrics-light.ipynb` và `03b-qwen-vl-metrics-heavy.ipynb` trên file predictions.

> Tip: bật smoke test bằng `SMOKE_TEST=1` để giới hạn số mẫu inference.


## 0) Chọn mode + paths

In [1]:
from pathlib import Path
import json

# =========================
# CHOOSE RUN_KIND
# =========================
RUN_ID = "B"  # baseline | A | B
if RUN_ID == "baseline":
    RUN_KIND = "baseline"  # "baseline" | "adapter"
else:
    RUN_KIND = "adapter"

# Dataset
DATASET_ID = "bbdontcry/vietnamese-image-captioning"

# -------------------------
# Mode 1: baseline (no finetune)
# -------------------------
BASELINE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# -------------------------
# Mode 2: adapter (finetuned)
# -------------------------
# Trỏ tới output của Notebook finetune (run_config.json + adapter/)
DATASET_PATH = Path(f"/kaggle/input/vn-textbook-qwen2vl-01-adapters/{RUN_ID}__Qwen2_VL_2B_Instruct")
RUN_CONFIG_PATH = DATASET_PATH / "run_config.json"
ADAPTER_DIR = DATASET_PATH / "adapter"

# Output folder
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load run_cfg if adapter mode
run_cfg = None
if RUN_KIND == "adapter":
    run_cfg = json.loads(Path(RUN_CONFIG_PATH).read_text(encoding="utf-8"))
    print("Loaded run_cfg from:", RUN_CONFIG_PATH)

print("RUN_KIND:", RUN_KIND)
print("OUT_DIR:", OUT_DIR)

# Derive RUN_ID for consistent prefix filenames
if RUN_KIND == "baseline":
    RUN_ID = "baseline"
elif RUN_KIND == "adapter":
    RUN_ID = str((run_cfg or {}).get("run_id") or (run_cfg or {}).get("preset") or "adapter").strip() or "adapter"
else:
    RUN_ID = str(RUN_KIND)

print("RUN_ID:", RUN_ID)

SMOKE_TEST = "0" # "1" để chạy smoke test
if SMOKE_TEST == "1":
    print("Running SMOKE TEST...")

Loaded run_cfg from: /kaggle/input/vn-textbook-qwen2vl-01-adapters/A__Qwen2_VL_2B_Instruct/run_config.json
RUN_KIND: adapter
OUT_DIR: /kaggle/working
RUN_ID: A


## 1) Install deps (inference only)

In [2]:
import sys, subprocess, json

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q"] + pkgs)

# Core
pip_install([
    "datasets>=2.18.0",
    "accelerate>=0.33.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "qwen-vl-utils>=0.0.11",
    "pandas>=2.0.0",
    "tqdm>=4.66.0",
])

# Transformers: Qwen2.5-VL đôi khi cần bản rất mới
need_source = False
if RUN_KIND == "adapter" and run_cfg is not None:
    need_source = ("Qwen2.5" in run_cfg.get("model_id",""))
if RUN_KIND == "baseline":
    need_source = False

if need_source:
    pip_install(["git+https://github.com/huggingface/transformers", "accelerate"])
else:
    pip_install(["transformers>=4.44.0"])

print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 52.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.7 MB/s eta 0:00:00
Done.


## 2) Load test split (detail only)

In [3]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID)
test_raw = raw["test"]
print("test len:", len(test_raw))
print("cols:", test_raw.column_names)

def pick_first(cols, candidates):
    for c in candidates:
        if c in cols: return c
    return None

img_col = pick_first(test_raw.column_names, ["image","img","image_pil"])
id_col  = pick_first(test_raw.column_names, ["id","sample_id"])
cap_detail_col = pick_first(test_raw.column_names, ["caption_detail","detail","caption_long","caption"])

print("img_col:", img_col, "| id_col:", id_col, "| cap_detail_col:", cap_detail_col)
assert img_col is not None
assert cap_detail_col is not None

README.md:   0%|          | 0.00/927 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/330M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/981 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/123 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

test len: 125
cols: ['id', 'image', 'caption_short', 'caption_detail', 'metadata_type', 'metadata_collection', 'metadata_title', 'metadata_grade', 'metadata_subject', 'metadata_author', 'metadata_publisher']
img_col: image | id_col: id | cap_detail_col: caption_detail


## 3) Load processor + model (baseline or adapter)

In [4]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig

def choose_amp_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, minor = torch.cuda.get_device_capability(0)
    # Kaggle T4 (7.5) -> FP16 only
    return torch.bfloat16 if major >= 8 else torch.float16

AMP_DTYPE = choose_amp_dtype()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("cap:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
print("AMP_DTYPE:", AMP_DTYPE)

# Pick model_id + prompts
if RUN_KIND == "baseline":
    model_id = BASELINE_MODEL_ID
    system_message = (
        "Bạn là một Vision-Language Model hỗ trợ người khiếm thị bằng cách tạo mô tả ảnh bằng tiếng Việt cho trang sách giáo khoa. "
        "Chỉ mô tả những gì thấy trong ảnh, không suy đoán."
        "Khi OCR, trích toàn bộ chữ nhìn thấy, theo thứ tự trên xuống dưới, trái sang phải"
    )

    prompt_detail = (
        "Hãy thuyết minh chi tiết trang SGK trong ảnh theo luồng đọc từ trên xuống dưới. "
        "Khi gặp chữ trong ảnh, hãy đưa vào đúng ngữ cảnh và trích nguyên văn trong ngoặc kép. "
        "Không suy luận."
    )
    gen_cfg = dict(max_new_tokens=2048, temperature=0.2, top_p=0.9, do_sample=False)

    # Vision tokens settings (fallback defaults)
    min_pixels = 256 * 28 * 28
    max_pixels = 1280 * 28 * 28
    quantization = None

else:
    model_id = run_cfg["model_id"]
    system_message = run_cfg.get("system_message","")
    prompt_detail  = run_cfg.get("prompt_detail","")
    gen = run_cfg.get("gen", {})
    gen_cfg = dict(
        max_new_tokens=int(gen.get("max_new_tokens_detail", 2048)),
        temperature=gen.get("temperature", 0.2),
        top_p=gen.get("top_p", 0.9),
        do_sample=gen.get("do_sample", False),
    )
    min_pixels = run_cfg.get("min_pixels", 256 * 28 * 28)
    max_pixels = run_cfg.get("max_pixels", 1280 * 28 * 28)
    quantization = run_cfg.get("quantization", None)

print("model_id:", model_id)
print("quantization:", quantization)

# Load processor (prefer saved processor if adapter mode)
processor_dir = None
if RUN_KIND == "adapter":
    cand = Path(RUN_CONFIG_PATH).parent / "processor"
    if cand.exists():
        processor_dir = cand

processor = AutoProcessor.from_pretrained(
    processor_dir if processor_dir else model_id,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
)

use_4bit = (quantization == "4bit")
bnb = None
if use_4bit:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=AMP_DTYPE,  # FP16 on T4
    )

if "Qwen2.5" in model_id:
    from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
else:
    from transformers import Qwen2VLForConditionalGeneration as ModelClass

base = ModelClass.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=AMP_DTYPE if not use_4bit else None,
    quantization_config=bnb,
)

if RUN_KIND == "adapter":
    from peft import PeftModel
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
else:
    model = base

model.eval()
print("Loaded model. Adapter:", RUN_KIND=="adapter")

2026-01-04 09:05:10.266500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767517510.445981      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767517510.495757      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767517510.914952      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767517510.914992      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767517510.914995      55 computation_placer.cc:177] computation placer alr

GPU: Tesla T4
cap: (7, 5)
AMP_DTYPE: torch.float16
model_id: Qwen/Qwen2-VL-2B-Instruct
quantization: None


`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Loaded model. Adapter: True


## 4) Inference + save predictions (detail only)

In [5]:
import pandas as pd
from tqdm.auto import tqdm

VISION_PREFIX = "<|vision_start|><|image_pad|><|vision_end|>\n"

@torch.inference_mode()
def generate_one(img, prompt_text: str):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": VISION_PREFIX + prompt_text},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        images=[img],
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=int(gen_cfg["max_new_tokens"]),
        do_sample=bool(gen_cfg["do_sample"]),
        temperature=float(gen_cfg["temperature"]),
        top_p=float(gen_cfg["top_p"]),
    )
    out_trim = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return processor.batch_decode(out_trim, skip_special_tokens=True)[0].strip()

TEST_LIMIT = 50  if "SMOKE_TEST" in globals() and SMOKE_TEST == "1" else None
ds = test_raw if TEST_LIMIT is None else test_raw.select(range(min(TEST_LIMIT, len(test_raw))))

rows = []
for ex in tqdm(ds, total=len(ds), desc=f"infer_{RUN_KIND}"):
    img = ex[img_col]
    gt = str(ex.get(cap_detail_col, ""))

    pred = generate_one(img, prompt_detail)

    row = {
        "id": ex.get(id_col, None) if id_col else None,
        "gt_detail": gt,
        "pred_detail": pred,
        "run_id": RUN_ID,
        "run_kind": RUN_KIND,
        "mode": RUN_KIND,
        "model_id": model_id,
        "adapter_dir": str(ADAPTER_DIR) if RUN_KIND=="adapter" else "",
    }
    rows.append(row)

pred_df = pd.DataFrame(rows)
pred_path = OUT_DIR / f"{RUN_ID}_predictions_test_detail.csv"
pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
print("Saved:", pred_path)
pred_df.head()

infer_adapter:   0%|          | 0/125 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

## 5) Save inference config (for reproducibility)

In [ ]:
import json, time
cfg_out = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_id": RUN_ID,
    "run_kind": RUN_KIND,
    "mode": RUN_KIND,
    "mode": RUN_KIND,
    "dataset_id": DATASET_ID,
    "model_id": model_id,
    "run_config_path": str(RUN_CONFIG_PATH) if RUN_KIND=="adapter" else "",
    "adapter_dir": str(ADAPTER_DIR) if RUN_KIND=="adapter" else "",
    "min_pixels": min_pixels,
    "max_pixels": max_pixels,
    "system_message": system_message,
    "prompt_detail": prompt_detail,
    "gen_cfg": gen_cfg,
}
cfg_path = OUT_DIR / f"{RUN_ID}_inference_config.json"
cfg_path.write_text(json.dumps(cfg_out, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", cfg_path)